<a href="https://colab.research.google.com/github/YassGan/3DGaussianSplatting-INRIA-Method-Colab/blob/feat%2Fworking_with_drive_ds/3DGaussianSplatting_INRIA_Method_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Making sure that we are using a GPU

In [1]:
!nvidia-smi

Thu Mar  6 02:07:54 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Inputting the zip file name (should be located in the drive) of the images which we want to create a 3d scene from

In [2]:
images_zip_file_name="RAW_Images"

# Installing pycolmap and colmap before any other installation or modificattion

In [3]:
# Install COLMAP in Colab
print("Installing COLMAP...")
!apt-get install -y colmap
!pip install pycolmap

Installing COLMAP...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libamd2 libcamd2 libccolamd2 libceres2 libcholmod3 libcolamd2 libcxsparse3 libevdev2
  libfreeimage3 libgflags2.2 libgoogle-glog0v5 libgudev-1.0-0 libinput-bin libinput10 libjxr0
  libmd4c0 libmetis5 libmtdev1 libqt5core5a libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5
  libqt5widgets5 libraw20 libspqr2 libsuitesparseconfig5 libwacom-bin libwacom-common libwacom9
  libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-render-util0 libxcb-util1 libxcb-xinerama0
  libxcb-xinput0 libxcb-xkb1 libxkbcommon-x11-0 qt5-gtk-platformtheme qttranslations5-l10n
Suggested packages:
  qt5-image-formats-plugins qtwayland5
The following NEW packages will be installed:
  colmap libamd2 libcamd2 libccolamd2 libceres2 libcholmod3 libcolamd2 libcxsparse3 libevdev2
  libfreeimage3 libgflags2.2 libgoogle-glog0v5 libgudev-1.0-0 libinput-bi

# Python downgrading


In [4]:
!wget -O mini.sh https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
!chmod +x mini.sh
!bash ./mini.sh -b -f -p /usr/local
!conda install -q -y python=3.7
import sys
sys.path.append('/usr/local/lib/python3.7/site-packages')
!python --version  # Should say Python 3.7.x

--2025-03-06 02:08:18--  https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 90665082 (86M) [application/x-sh]
Saving to: ‘mini.sh’

mini.sh             100%[===================>]  86.46M  83.8MB/s    in 1.0s    

2025-03-06 02:08:19 (83.8 MB/s) - ‘mini.sh’ saved [90665082/90665082]

PREFIX=/usr/local
Unpacking payload ...
                                                                                 
Installing base environment...





Preparing transaction: - \ | / - \ done
Executing transaction: / - \ | / - \ | / - \ | / - \ | / - \ | / done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected beha

# CUDA 11.8

In [5]:
!wget https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
!chmod +x cuda_11.8.0_520.61.05_linux.run
!./cuda_11.8.0_520.61.05_linux.run --silent --toolkit --no-drm --no-man-page
import os
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia'
!nvcc --version  # Should show CUDA 11.8

--2025-03-06 02:08:50--  https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.223.211.50, 23.223.211.89
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.223.211.50|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4336730777 (4.0G) [application/octet-stream]
Saving to: ‘cuda_11.8.0_520.61.05_linux.run’

cuda_11.8.0_520.61. 100%[===================>]   4.04G   120MB/s    in 40s     

2025-03-06 02:09:30 (104 MB/s) - ‘cuda_11.8.0_520.61.05_linux.run’ saved [4336730777/4336730777]

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0


#Pytorch with cuda

In [6]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==1.12.1+cu116 torchvision==0.13.1+cu116 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu116

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu116
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 592.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 39.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 26.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 85.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 47.4 MB/s eta 0:00:00


# Verification of the versions

In [7]:
import torch
print(torch.cuda.is_available())  # Should be True
print(torch.version.cuda)        # Should be 11.3 (from PyTorch)
print(torch.cuda.get_device_name(0))  # Should show GPU

True
12.4
Tesla T4


In [8]:
##making sure that we are always using the GPU and not a CPU
!nvidia-smi

Thu Mar  6 02:14:57 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Cloning the 3D Gaussian Splatting algorithm and the submodules of the algorithm

In [9]:
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile

%cd /content/gaussian-splatting
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/gaussian-splatting/submodules/simple-knn


/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 603, done.
remote: Total 603 (delta 0), reused 0 (delta 0), pack-reused 603 (from 1)
Receiving objects: 100% (603/603), 2.09 MiB | 4.08 MiB/s, done.
Resolving deltas: 100% (349/349), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects: 100% (174/174), done.        
remote: Total 3293 (delta 171), reused 280 (delta 148), pack-reused 2971 (from 1)        
Receiving objects: 

# Mounting the drive

In [10]:


import os
import zipfile
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)
# Paths




Mounted at /content/drive


# Choosing the file

In [11]:
!unzip "/content/drive/MyDrive/{images_zip_file_name}.zip" -d "/content/{images_zip_file_name}"



Archive:  /content/drive/MyDrive/RAW_Images.zip
   creating: /content/RAW_Images/RAW_Images/
  inflating: /content/RAW_Images/RAW_Images/DSC05572.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05573.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05574.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05575.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05576.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05577.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05578.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05579.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05580.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05581.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05582.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05583.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05584.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05585.jpg  
  inflating: /content/RAW_Images/RAW_Images/DSC05586.jpg  
  inflating: /content/

COLMAP Work with RAW images zip file

In [12]:
from pathlib import Path
import os
import subprocess
import pycolmap
from google.colab import files
import time



print("---------------->",images_zip_file_name)



# Define paths
output_path = Path("/content/COLMAP_Output")  # Base output directory (absolute path for clarity)
sparse_output_path = output_path / "sparse"  # Final sparse reconstruction directory
image_dir = Path(f"/content/{images_zip_file_name}/{images_zip_file_name}")  # Input images directory
initial_recon_path = output_path / "initial"  # Temporary directory for initial reconstruction
database_path = output_path / "database.db"
undistorted_output = output_path / "undistorted"

# Create and verify directories
print("Creating directories...")
output_path.mkdir(exist_ok=True)
sparse_output_path.mkdir(exist_ok=True)
initial_recon_path.mkdir(exist_ok=True)
print(f"Checking directories after creation:")
print(f" - {output_path}: {'exists' if output_path.exists() else 'not created'}")
print(f" - {sparse_output_path}: {'exists' if sparse_output_path.exists() else 'not created'}")
print(f" - {initial_recon_path}: {'exists' if initial_recon_path.exists() else 'not created'}")
!ls -l /content/COLMAP_Output  # List contents to confirm in terminal

# Verify image directory
print(f"Image directory: {image_dir}")
print(f" - Exists: {image_dir.exists()}")
print(f" - Contents: {list(image_dir.glob('*'))}")
if not image_dir.exists() or not list(image_dir.glob('*')):
    raise ValueError("Image directory is empty or doesn’t exist. Check your zip file structure.")

# Step 1: Initial SfM pipeline
print("Running initial feature extraction...")
pycolmap.extract_features(database_path, image_dir)
print(f"Database created: {database_path.exists()}")
!ls -l /content/COLMAP_Output

print("Running initial feature matching...")
pycolmap.match_exhaustive(database_path)

print("Running initial incremental mapping...")
maps = pycolmap.incremental_mapping(database_path, image_dir, initial_recon_path)
maps[0].write(initial_recon_path)
print(f"Initial reconstruction contents: {list(initial_recon_path.glob('*'))}")
!ls -l /content/COLMAP_Output/initial

# Step 2: Undistort images
print("Running image undistortion...")
subprocess.run([
    "colmap", "image_undistorter",
    "--image_path", str(image_dir),
    "--input_path", str(initial_recon_path),
    "--output_path", str(undistorted_output),
    "--output_type", "COLMAP",
    "--max_image_size", "2000"
], check=True)
print(f"Undistorted directory contents: {list(undistorted_output.glob('*'))}")
!ls -l /content/COLMAP_Output/undistorted

# Update paths
undistorted_image_dir = undistorted_output / "images"
print(f"Undistorted images directory: {undistorted_image_dir}")
print(f" - Contents: {list(undistorted_image_dir.glob('*'))}")

# Step 3: Re-run SfM on undistorted images
print("Extracting features from undistorted images...")
pycolmap.extract_features(database_path, undistorted_image_dir)

print("Matching features from undistorted images...")
pycolmap.match_exhaustive(database_path)

print("Running incremental mapping on undistorted images...")
maps = pycolmap.incremental_mapping(database_path, undistorted_image_dir, sparse_output_path)
maps[0].write(sparse_output_path)
print(f"Final sparse output contents: {list(sparse_output_path.glob('*'))}")
!ls -l /content/COLMAP_Output/sparse

# Step 4: Verify camera models
print("Verifying camera models in the output...")
reconstruction = pycolmap.Reconstruction(sparse_output_path)
for cam_id, cam in reconstruction.cameras.items():
    print(cam)

print("COLMAP processing completed.")
print("Refresh the file explorer (right-click > Refresh) to see COLMAP_Output.")

----------------> RAW_Images
Creating directories...
Checking directories after creation:
 - /content/COLMAP_Output: exists
 - /content/COLMAP_Output/sparse: exists
 - /content/COLMAP_Output/initial: exists
total 8
drwxr-xr-x 2 root root 4096 Mar  6 02:19 initial
drwxr-xr-x 2 root root 4096 Mar  6 02:19 sparse
Image directory: /content/RAW_Images/RAW_Images
 - Exists: True
 - Contents: [PosixPath('/content/RAW_Images/RAW_Images/DSC05578.jpg'), PosixPath('/content/RAW_Images/RAW_Images/DSC05573.jpg'), PosixPath('/content/RAW_Images/RAW_Images/DSC05602.jpg'), PosixPath('/content/RAW_Images/RAW_Images/DSC05580.jpg'), PosixPath('/content/RAW_Images/RAW_Images/DSC05579.jpg'), PosixPath('/content/RAW_Images/RAW_Images/DSC05581.jpg'), PosixPath('/content/RAW_Images/RAW_Images/DSC05584.jpg'), PosixPath('/content/RAW_Images/RAW_Images/DSC05572.jpg'), PosixPath('/content/RAW_Images/RAW_Images/DSC05596.jpg'), PosixPath('/content/RAW_Images/RAW_Images/DSC05595.jpg'), PosixPath('/content/RAW_Images

In [13]:
# Step 4: Verify camera models
print("Verifying camera models in the output...")
reconstruction = pycolmap.Reconstruction(sparse_output_path)
for cam_id, cam in reconstruction.cameras.items():
    print(cam)

Verifying camera models in the output...
Camera(camera_id=33, model=SIMPLE_RADIAL, width=1264, height=832, params=[1166.8, 632, 416, 0.0206223] (f, cx, cy, k))
Camera(camera_id=32, model=SIMPLE_RADIAL, width=1264, height=832, params=[1125.69, 632, 416, 0.00595257] (f, cx, cy, k))
Camera(camera_id=31, model=SIMPLE_RADIAL, width=1264, height=832, params=[1040.28, 632, 416, -0.000654443] (f, cx, cy, k))
Camera(camera_id=30, model=SIMPLE_RADIAL, width=1264, height=832, params=[1037.48, 632, 416, 0.00306093] (f, cx, cy, k))
Camera(camera_id=13, model=SIMPLE_RADIAL, width=1264, height=832, params=[1056.96, 632, 416, 0.00837125] (f, cx, cy, k))
Camera(camera_id=12, model=SIMPLE_RADIAL, width=1264, height=832, params=[1053.36, 632, 416, 0.00063098] (f, cx, cy, k))
Camera(camera_id=11, model=SIMPLE_RADIAL, width=1264, height=832, params=[1051.06, 632, 416, 0.00481582] (f, cx, cy, k))
Camera(camera_id=10, model=SIMPLE_RADIAL, width=1264, height=832, params=[1046.7, 632, 416, 0.00571112] (f, cx, 

# Arranging Input folder for the 3D Gaussian INRIA

In [21]:
import os
import shutil
from pathlib import Path

# Ensure output_path is defined
output_path = Path("/content/COLMAP_Output")

# Correct path checking using Path
undisorted_sparse_path = output_path / "undistorted"
print(f" - {undisorted_sparse_path}: {'exists' if undisorted_sparse_path.exists() else 'not created'}")

# Define paths correctly
input_images_folder = output_path / "undistorted" / "images"  # Folder with input images
colmap_output_folder = output_path / "undistorted" / "sparse"  # Folder containing COLMAP's sparse output
new_parent_folder = "3D_Gaussian_Splatting_input_folder2"  # New folder to create

# Create the new parent folder
new_parent = Path(f"/content/{new_parent_folder}")
new_parent.mkdir(parents=True, exist_ok=True)

# 1. Copy input images to "images" subfolder
images_subfolder = new_parent / "images"
images_subfolder.mkdir(exist_ok=True)

# Check if input images folder exists before copying
if not input_images_folder.exists():
    print(f"Warning: Input images folder does not exist: {input_images_folder}")
else:
    for img in input_images_folder.glob("*"):
        if img.is_file() and img.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            shutil.copy(img, images_subfolder / img.name)

# 2. Create "sparse" subfolder and copy COLMAP output files
sparse_subfolder = new_parent / "sparse/0/"
sparse_subfolder.parent.mkdir(parents=True, exist_ok=True)  # Create 'sparse' parent directory
sparse_subfolder.mkdir(exist_ok=True)  # Create '0' subfolder

# Copy COLMAP reconstruction files (cameras.bin, images.bin, points3D.bin)
required_colmap_files = ["cameras.bin", "images.bin", "points3D.bin"]

# Check if COLMAP sparse folder exists before copying
if not colmap_output_folder.exists():
    print(f"Warning: COLMAP sparse folder does not exist: {colmap_output_folder}")
else:
    for file in required_colmap_files:
        src = colmap_output_folder / file
        if src.exists():
            shutil.copy(src, sparse_subfolder / file)
        else:
            print(f"Warning: {file} not found in COLMAP output folder!")

print(f"✅ Dataset folder created at: {new_parent_folder}")


 - /content/COLMAP_Output/undistorted: exists
✅ Dataset folder created at: 3D_Gaussian_Splatting_input_folder2


# Training the model

In [22]:
!python /content/gaussian-splatting/train.py \
  -s /content/3D_Gaussian_Splatting_input_folder2// \
  -m /content/output \
  --iterations 30000 \
  --test_iterations 1000 2000 3000 4000 5000 \
  --save_iterations 1000 2000 3000 4000 5000


Optimizing /content/output
Output folder: /content/output [06/03 02:31:57]
Tensorboard not available: not logging progress [06/03 02:31:57]
Reading camera 33/33 [06/03 02:31:58]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [06/03 02:31:58]
Loading Training Cameras [06/03 02:31:58]
Traceback (most recent call last):
  File "/content/gaussian-splatting/train.py", line 216, in <module>
    training(lp.extract(args), op.extract(args), pp.extract(args), args.test_iterations, args.save_iterations, args.checkpoint_iterations, args.start_checkpoint, args.debug_from)
  File "/content/gaussian-splatting/train.py", line 35, in training
    scene = Scene(dataset, gaussians)
  File "/content/gaussian-splatting/scene/__init__.py", line 73, in __init__
    self.train_cameras[resolution_scale] = cameraList_from_camInfos(scene_info.train_cameras, resolution_scale, args)
  File "/content/gaussian-splatting/utils/camera_utils.py", line 58, in cameraList_from_camInfo